In [1]:
# Final Classifier model - rf and xgboost 
# For each cluster we train and tune a rf model that classifies rain into three categories: 1. No (<0.1 mm)
# 2. Light (<= 8.4 mm)
# 3. Heavy (> 8.4 mm)

# Workflow:
# 1. Load the complete df, format and create a prec intensity column (target)
# 2. Split df by clusters and run a for loop
# 3. Convert into np array for sklearn fit
# 4. Fit and tune a random forest using cv
# 5. Get validation metrics from training stations
# 6. Get validation metrics from validation stations 
# 7. Save trained model using joblib
# 8. Visualize final validation metrics for all clusters

In [2]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

train_data_path = r"..\Data\Processed\Classification_Regression\final_train_df.csv"
test_data_path  = r"..\Data\Processed\Classification_Regression\final_test_df.csv"
class_metrics_dir = r"..\Data\Processed\Classification_Regression\Classification_metrics"
class_models_dir  = r"..\Data\Processed\Classification_Regression\Classification_models"  

os.makedirs(class_metrics_dir, exist_ok=True)
os.makedirs(class_models_dir,  exist_ok=True)

random_state_num = 64
target_variable  = "intensity_prec"
n_cores          = 1 

In [3]:
# %%
train_df = pd.read_csv(train_data_path)
test_df  = pd.read_csv(test_data_path)   

season_dict = {'winter': 1,
                'pre_monsoon': 2, 
                'monsoon': 3,
                'post_monsoon': 4            
                }

train_df['season_num'] = train_df['season'].map(season_dict)
test_df['season_num'] = test_df['season'].map(season_dict)

conditions_train = [
    (train_df['insitu_prec'] < 0.1),
    (train_df['insitu_prec'] >= 0.1) 
]
conditions_test = [
    (test_df['insitu_prec'] < 0.1),
    (test_df['insitu_prec'] >= 0.1) ]

intensity_class = [0, 1]

train_df[target_variable] = np.select(conditions_train, intensity_class, default=np.nan)
test_df[target_variable]  = np.select(conditions_test,  intensity_class, default=np.nan)  # was using undefined `conditions`

cluster_list = sorted(train_df['cluster'].unique().tolist())
print(f"Clusters found: {cluster_list}")

Clusters found: [0.0, 1.0, 2.0, 3.0]


In [4]:
# %%
def get_param_grids():
    param_dist_rf = {
        'n_estimators':     [50, 100, 200, 500, 1000],
        'max_features':     ['sqrt', 'log2'],
        'max_depth':        [None, 10, 20, 30, 50],
        'min_samples_split':[2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }
    param_dist_xgb = {
        'n_estimators':  [50, 100, 200, 500],
        'max_depth':     [3, 5, 7, 10],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'subsample':     [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'gamma':         [0, 0.1, 0.5]
    }
    return param_dist_rf, param_dist_xgb


def fit_and_evaluate(model_type, X_train, y_train, X_test, y_test,
                     cluster, random_state_num, n_cores):
    """
    Fits RF or XGBoost classifier with RandomizedSearchCV.
    Returns best model, train accuracy, test accuracy, best params.
    """
    param_dist_rf, param_dist_xgb = get_param_grids()

    if model_type == 'rf':
        base_model  = RandomForestClassifier(random_state=random_state_num, n_jobs=n_cores)
        param_dist  = param_dist_rf
    elif model_type == 'xgb':
        base_model  = XGBClassifier(random_state=random_state_num, n_jobs=n_cores,
                                    eval_metric='mlogloss', verbosity=0)
        param_dist  = param_dist_xgb
    else:
        raise ValueError(f"model_type must be 'rf' or 'xgb', got '{model_type}'")

    search = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=param_dist,
        n_iter=50,
        cv=5,
        verbose=1,
        random_state=random_state_num,
        n_jobs=n_cores,
        scoring='accuracy'
    )
    search.fit(X_train, y_train)
    best_model = search.best_estimator_

    train_preds    = best_model.predict(X_train)
    test_preds     = best_model.predict(X_test)
    train_accuracy = accuracy_score(y_train, train_preds)
    test_accuracy  = accuracy_score(y_test,  test_preds)

    print(f"\n[Cluster {cluster} | {model_type.upper()}]")
    print(f"  Best params      : {search.best_params_}")
    print(f"  CV accuracy      : {search.best_score_:.4f}")
    print(f"  Train accuracy   : {train_accuracy:.4f}")
    print(f"  Test accuracy    : {test_accuracy:.4f}")
    print(classification_report(y_test, test_preds,
                                target_names=['No Rain','Light']))

    return best_model, train_accuracy, test_accuracy, search.best_params_

In [5]:
# %%
results = []  

for cluster in cluster_list:
    print(f"\n{'='*55}\nProcessing Cluster {cluster}\n{'='*55}")

    # 2. Split by cluster
    cluster_train_df = train_df[train_df['cluster'] == cluster].copy()
    cluster_test_df  = test_df[test_df['cluster']  == cluster].copy()
    cluster_train_df = cluster_train_df.dropna(subset=[target_variable])
    cluster_test_df  = cluster_test_df.dropna(subset=[target_variable])

    # 3. Build feature matrices — drop non-feature columns
    cols_to_drop = [target_variable, 'cluster', 'date', 'insitu_prec', 'season']
    X_train = cluster_train_df.drop(columns=cols_to_drop).values
    X_test  = cluster_test_df.drop(columns=cols_to_drop).values
    y_train = cluster_train_df[target_variable].values.astype(int)
    y_test  = cluster_test_df[target_variable].values.astype(int)

    feature_list = cluster_train_df.drop(columns=cols_to_drop).columns.tolist()

    for model_type in ['rf', 'xgb']:
        # 4 + 5 + 6. Fit, tune, evaluate
        best_model, train_acc, test_acc, best_params = fit_and_evaluate(
            model_type, X_train, y_train, X_test, y_test,
            cluster, random_state_num, n_cores
        )

        # 7. Save model
        model_filename = f"cluster{cluster}_{model_type}_classifier_2classes.pkl"
        model_path     = os.path.join(class_models_dir, model_filename)
        joblib.dump(best_model, model_path)
        print(f"  Model saved → {model_path}")

        # Collect metrics
        results.append({
            'cluster':        cluster,
            'model':          model_type.upper(),
            'train_accuracy': round(train_acc, 4),
            'test_accuracy':  round(test_acc,  4),
        })


Processing Cluster 0.0
Fitting 5 folds for each of 50 candidates, totalling 250 fits

[Cluster 0.0 | RF]
  Best params      : {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 10}
  CV accuracy      : 0.8882
  Train accuracy   : 0.9074
  Test accuracy    : 0.8709
              precision    recall  f1-score   support

     No Rain       0.88      0.96      0.92      7960
       Light       0.83      0.61      0.70      2658

    accuracy                           0.87     10618
   macro avg       0.85      0.78      0.81     10618
weighted avg       0.87      0.87      0.86     10618

  Model saved → ..\Data\Processed\Classification_Regression\Classification_models\cluster0.0_rf_classifier_2classes.pkl
Fitting 5 folds for each of 50 candidates, totalling 250 fits

[Cluster 0.0 | XGB]
  Best params      : {'subsample': 0.6, 'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.05, 'gamma': 0.1, 'colsample_bytree': 1.0}
  CV accur

In [6]:
# %%
# 8. Validation metrics table for all clusters
results_df = pd.DataFrame(results)

# Averages row per model type
avg_rows = (
    results_df.groupby('model')[['train_accuracy', 'test_accuracy']]
    .mean()
    .round(4)
    .reset_index()
    .assign(cluster='AVERAGE')
)

summary_df = pd.concat([results_df, avg_rows], ignore_index=True)
summary_df = summary_df[['cluster', 'model', 'train_accuracy', 'test_accuracy']]

print("\n===== Validation Metrics Summary =====")
print(summary_df.to_string(index=False))

# Save to CSV
summary_path = os.path.join(class_metrics_dir, "validation_metrics_summary_2classes.csv")
summary_df.to_csv(summary_path, index=False)
print(f"\nSummary saved → {summary_path}")


===== Validation Metrics Summary =====
cluster model  train_accuracy  test_accuracy
    0.0    RF          0.9074         0.8709
    0.0   XGB          0.8935         0.8681
    1.0    RF          0.9494         0.8633
    1.0   XGB          0.8862         0.8523
    2.0    RF          0.9462         0.9135
    2.0   XGB          0.9334         0.9130
    3.0    RF          0.9642         0.8680
    3.0   XGB          0.8587         0.8709
AVERAGE    RF          0.9418         0.8789
AVERAGE   XGB          0.8930         0.8761

Summary saved → ..\Data\Processed\Classification_Regression\Classification_metrics\validation_metrics_summary_2classes.csv
